# AP07 - Introduction to DSPy

### Program LLMs like software, not prompts

**Duration:** 2.5 hours

**What you'll learn:**
- What is DSPy and why it exists
- DSPy vs traditional prompting
- Signatures: declaring what you want
- Modules: reusable LLM components
- ChainOfThought for reasoning
- Building your first DSPy program
- When to use DSPy

**Prerequisites:**
- Understanding of prompt engineering
- Experience with LLMs (GPT-4, Claude, etc.)
- Basic Python

---

## 🔧 Setup

In [ ]:
# Install DSPy
# !pip install -qU dspy-ai openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import dspy

load_dotenv()

# Configure DSPy with OpenAI
lm = dspy.OpenAI(model="gpt-4o-mini", max_tokens=500)
dspy.settings.configure(lm=lm)

print(f"✅ DSPy configured with {lm.kwargs['model']}")
print(f"   DSPy version: {dspy.__version__}")

---

## 1. What is DSPy?

### The Problem with Traditional Prompting

```python
# Traditional approach
prompt = f"""
You are a helpful assistant. Answer the question based on the context.

Context: {context}
Question: {question}

Let's think step by step:
1. First, analyze the context
2. Then, identify key information
3. Finally, provide a clear answer

Answer:
"""

# Problems:
# - Brittle: Break with different models
# - Manual: Tweaking prompts for each task
# - Not reusable: Can't compose easily
# - Hard to optimize: No systematic improvement
```

### The DSPy Solution

```python
# DSPy approach
class QA(dspy.Signature):
    """Answer question based on context"""
    context = dspy.InputField()
    question = dspy.InputField()
    answer = dspy.OutputField()

qa = dspy.ChainOfThought(QA)

# Benefits:
# ✅ Declarative: Say what you want, not how
# ✅ Composable: Build complex programs from modules
# ✅ Optimizable: Automatic prompt optimization
# ✅ Portable: Works across different LLMs
```

### Key Philosophy

> **"Programming, not prompting"**
> 
> DSPy treats prompts as parameters to be optimized, not hand-crafted strings.

---

## 2. Core Concepts

### 1. Signatures: Declaring What You Want

Signatures are like function signatures for LLMs.

```
InputField(s) → LLM → OutputField(s)
```

In [ ]:
import dspy

# Simple signature: question → answer
class BasicQA(dspy.Signature):
    """Answer questions with short responses"""
    question = dspy.InputField()
    answer = dspy.OutputField(desc="short answer, 1-2 sentences")

# Use it
qa = dspy.Predict(BasicQA)
response = qa(question="What is Python?")

print("Question:", "What is Python?")
print("Answer:", response.answer)

### Inline Signatures (Shorthand)

In [ ]:
# Instead of defining a class, use string format
qa_inline = dspy.Predict("question -> answer")

response = qa_inline(question="What is machine learning?")
print("Inline signature answer:", response.answer)

# More complex inline
summarizer = dspy.Predict("long_text -> summary")
sentiment = dspy.Predict("text -> sentiment: positive, negative, or neutral")

print("\n✅ Inline signatures are quick and convenient!")

### 2. Modules: Reusable Components

Modules are DSPy's building blocks.

**Built-in modules:**
- `dspy.Predict` - Basic prediction
- `dspy.ChainOfThought` - Reasoning before answer
- `dspy.ReAct` - Reasoning + Acting (for agents)
- `dspy.ProgramOfThought` - Code generation for reasoning

---

## 3. ChainOfThought - Reasoning Module

In [ ]:
# Define signature
class MathProblem(dspy.Signature):
    """Solve math word problems"""
    problem = dspy.InputField()
    answer = dspy.OutputField(desc="final numerical answer")

# Use ChainOfThought for reasoning
cot = dspy.ChainOfThought(MathProblem)

# Test
problem = "If Alice has 5 apples and buys 3 more, then gives 2 to Bob, how many does she have?"
result = cot(problem=problem)

print("Problem:", problem)
print("\nReasoning:", result.rationale)  # Shows thinking process
print("\nAnswer:", result.answer)

### ChainOfThought vs Predict

In [ ]:
# Compare both approaches
predict_module = dspy.Predict(MathProblem)
cot_module = dspy.ChainOfThought(MathProblem)

problem = "A train travels 60 km in 1 hour. How far in 2.5 hours?"

print("=" * 70)
print("Using Predict (direct answer):")
print("=" * 70)
pred_result = predict_module(problem=problem)
print("Answer:", pred_result.answer)

print("\n" + "=" * 70)
print("Using ChainOfThought (reasoning first):")
print("=" * 70)
cot_result = cot_module(problem=problem)
print("Reasoning:", cot_result.rationale)
print("Answer:", cot_result.answer)

print("\n💡 ChainOfThought provides transparency and often better accuracy!")

---

## 4. Building Complex Programs

### Composing Multiple Modules

In [ ]:
class MultiStepQA(dspy.Module):
    def __init__(self):
        super().__init__()
        
        # Step 1: Understand the question
        self.understand = dspy.ChainOfThought("question -> key_concepts")
        
        # Step 2: Generate answer
        self.answer = dspy.ChainOfThought("question, key_concepts -> answer")
        
        # Step 3: Verify answer
        self.verify = dspy.Predict("question, answer -> confidence: high, medium, or low")
    
    def forward(self, question):
        # Execute pipeline
        concepts = self.understand(question=question)
        answer = self.answer(question=question, key_concepts=concepts.key_concepts)
        confidence = self.verify(question=question, answer=answer.answer)
        
        return dspy.Prediction(
            concepts=concepts.key_concepts,
            answer=answer.answer,
            reasoning=answer.rationale,
            confidence=confidence.confidence
        )

# Test the pipeline
multi_qa = MultiStepQA()
result = multi_qa(question="Why is the sky blue?")

print("Question: Why is the sky blue?")
print("\nKey Concepts:", result.concepts)
print("\nReasoning:", result.reasoning)
print("\nAnswer:", result.answer)
print("\nConfidence:", result.confidence)

---

## 5. Practical Example: RAG with DSPy

In [ ]:
class RAGSignature(dspy.Signature):
    """Answer question based on retrieved context"""
    context = dspy.InputField(desc="relevant information from documents")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="answer citing the context")

class SimpleRAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.generate_answer = dspy.ChainOfThought(RAGSignature)
    
    def forward(self, context, question):
        return self.generate_answer(context=context, question=question)

# Test RAG
rag = SimpleRAG()

context = """
Python was created by Guido van Rossum and first released in 1991.
It emphasizes code readability with significant whitespace.
Python supports multiple programming paradigms including procedural,
object-oriented, and functional programming.
"""

question = "When was Python created and by whom?"

result = rag(context=context, question=question)

print("Question:", question)
print("\nReasoning:", result.rationale)
print("\nAnswer:", result.answer)

---

## 6. Comparison: Traditional vs DSPy

### Traditional Prompting

In [ ]:
from openai import OpenAI

client = OpenAI()

# Traditional approach - manual prompt engineering
def traditional_qa(question, context):
    prompt = f"""
You are a helpful assistant. Answer the question based on the context.

Context: {context}

Question: {question}

Provide a clear, concise answer:
"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )
    
    return response.choices[0].message.content

# Test
answer = traditional_qa(
    question="What is Python?",
    context="Python is a high-level programming language known for readability."
)

print("Traditional approach:")
print(answer)

### DSPy Approach

In [ ]:
# DSPy approach - declarative and optimizable
dspy_qa = dspy.ChainOfThought("context, question -> answer")

result = dspy_qa(
    context="Python is a high-level programming language known for readability.",
    question="What is Python?"
)

print("DSPy approach:")
print(result.answer)

print("\n" + "="*70)
print("Key Differences:")
print("="*70)
print("Traditional: Manual prompt, brittle, hard to optimize")
print("DSPy: Declarative, composable, auto-optimizable")

---

## 7. Advanced: Multi-Hop Reasoning

In [ ]:
class MultiHopQA(dspy.Module):
    def __init__(self):
        super().__init__()
        # Break down complex question
        self.decompose = dspy.ChainOfThought(
            "complex_question -> sub_questions: list of simpler questions"
        )
        # Answer each sub-question
        self.answer_sub = dspy.ChainOfThought("question -> answer")
        # Synthesize final answer
        self.synthesize = dspy.ChainOfThought(
            "sub_answers, original_question -> final_answer"
        )
    
    def forward(self, question):
        # Step 1: Decompose
        decomposition = self.decompose(complex_question=question)
        
        # Step 2: Answer sub-questions
        sub_answers = []
        for sub_q in decomposition.sub_questions.split("; "):
            if sub_q.strip():
                ans = self.answer_sub(question=sub_q)
                sub_answers.append(f"Q: {sub_q}\nA: {ans.answer}")
        
        # Step 3: Synthesize
        final = self.synthesize(
            sub_answers="\n\n".join(sub_answers),
            original_question=question
        )
        
        return dspy.Prediction(
            sub_questions=decomposition.sub_questions,
            sub_answers=sub_answers,
            final_answer=final.final_answer
        )

# Test
multi_hop = MultiHopQA()
result = multi_hop(
    question="If Python was created in 1991 and JavaScript in 1995, which is older and by how many years?"
)

print("Complex Question:")
print("If Python was created in 1991 and JavaScript in 1995, which is older and by how many years?")
print("\nSub-questions:")
print(result.sub_questions)
print("\nFinal Answer:")
print(result.final_answer)

---

## 8. When to Use DSPy

### ✅ Use DSPy When:

1. **Building Complex Pipelines**
   - Multiple steps with dependencies
   - Need composability
   - Want to optimize end-to-end

2. **Working Across Models**
   - Same logic, different LLMs
   - Want portability
   - Need model-agnostic code

3. **Need Optimization**
   - Want to improve prompts systematically
   - Have training data
   - Need reproducible improvements

4. **Building Reusable Components**
   - Creating libraries
   - Multiple similar tasks
   - Want abstractions

### ❌ Skip DSPy When:

1. **Simple One-Off Tasks**
   - Single prompt is enough
   - No need for optimization
   - Quick prototype

2. **Highly Custom Prompts**
   - Need very specific formatting
   - Unusual prompt structure
   - LangChain is better here

3. **Learning LLMs**
   - Start with basic prompting first
   - Understand fundamentals
   - Then adopt DSPy

---

## 9. DSPy vs Other Frameworks

| Feature | DSPy | LangChain | LlamaIndex |
|---------|------|-----------|------------|
| **Focus** | Prompt optimization | Workflow orchestration | Data indexing |
| **Approach** | Declarative, compiled | Imperative chains | Simple API |
| **Optimization** | Built-in | Manual | Limited |
| **Learning Curve** | Medium | Medium | Low |
| **Best For** | Optimized pipelines | Complex agents | RAG apps |
| **Composability** | Excellent | Good | Limited |

**When to combine:**
- DSPy for prompt optimization
- LangChain for agent orchestration
- LlamaIndex for data retrieval

---

## 🎯 Practice Exercises

### Exercise 1: Custom Signature
Create a DSPy signature for:
- Sentiment analysis (text → sentiment, confidence)
- Test with both Predict and ChainOfThought
- Compare outputs

### Exercise 2: Multi-Step Pipeline
Build a module that:
1. Extracts entities from text
2. Classifies entity types
3. Generates summary with entities

### Exercise 3: RAG Enhancement
Extend SimpleRAG to:
- Generate follow-up questions
- Assess answer quality
- Provide confidence scores

### Exercise 4: Comparison Study
Compare DSPy vs traditional prompting:
- Same task, both approaches
- Measure accuracy
- Evaluate maintainability

### Challenge: Smart Summarizer
Build a module that:
- Takes long text
- Identifies key points
- Generates summaries at different lengths
- Verifies accuracy

---

## 📚 Key Takeaways

1. **DSPy = Programming LLMs** - Declarative, not imperative
2. **Signatures** - Declare inputs and outputs
3. **Modules** - Reusable components (Predict, ChainOfThought, ReAct)
4. **Composability** - Build complex from simple
5. **Optimization** - Prompts as parameters (covered in AP08)
6. **Portability** - Works across different LLMs
7. **Maintainability** - Cleaner code than string templates

---

## 🚀 Next Steps

**AP08 - DSPy Optimizers**
- Automatic prompt optimization
- BootstrapFewShot
- MIPRO optimizer
- Training and evaluation

**Resources:**
- [DSPy Documentation](https://dspy-docs.vercel.app/)
- [DSPy GitHub](https://github.com/stanfordnlp/dspy)
- [DSPy Examples](https://github.com/stanfordnlp/dspy/tree/main/examples)

---

*Created as part of the Applied AI module*  
*Last updated: January 2025*